# Week 2: Preprocessing — Cloud Masking & Water Index
### Lagos Barrier Island Shoreline Change Detection Project

**Purpose:** Apply cloud/shadow masking (Sentinel-2 SCL band) and compute MNDWI for each year's selected scene. Builds on `01_data_acquisition.ipynb`; re-fetches scene selection here (a fast, server-side query) so this notebook is self-contained.

MNDWI = (Green − SWIR) / (Green + SWIR) — chosen over plain NDWI for better suppression of false positives from Lagos's dense built-up areas.

## 1. Authenticate & Initialize Earth Engine

In [ ]:
import ee

ee.Authenticate(
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.full_control",
        "https://www.googleapis.com/auth/drive",
    ],
)
ee.Initialize()


## 2. Imports & AOI

In [ ]:
from pipeline_utils import get_best_scene, mask_clouds_scl, compute_mndwi
import geemap

aoi = ee.Geometry.Rectangle([3.30, 6.38, 3.55, 6.48])


## 3. Re-select scenes (fast server-side query, not a re-download)

In [ ]:
years = list(range(2017, 2026))

selected_scenes = {}
for yr in years:
    scene = get_best_scene(yr, aoi)
    if scene is not None:
        selected_scenes[yr] = scene

print(f"Scenes available: {len(selected_scenes)} / {len(years)}")


## 4. Cloud masking

In [ ]:
masked_scenes = {
    yr: mask_clouds_scl(img) for yr, img in selected_scenes.items()
}
print(f"Cloud masking applied to {len(masked_scenes)} scenes.")


## 5. MNDWI computation

In [ ]:
mndwi_scenes = {
    yr: compute_mndwi(img) for yr, img in masked_scenes.items()
}
print(f"MNDWI computed for {len(mndwi_scenes)} scenes.")


## 6. Visual QA — MNDWI for a sample year

In [ ]:
Map2 = geemap.Map(center=[6.43, 3.42], zoom=11)

mndwi_vis = {
    "bands": ["MNDWI"],
    "min": -0.5,
    "max": 0.5,
    "palette": ["brown", "white", "blue"],
}

sample_year = 2020
Map2.addLayer(mndwi_scenes[sample_year], mndwi_vis, f"MNDWI {sample_year}")
Map2.addLayer(aoi, {}, "AOI", opacity=0.3)
Map2
